In [1]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPU devices available:", tf.config.list_physical_devices('GPU'))
print("TF built with CUDA:", tf.sysconfig.get_build_info()["cuda_version"])

import glob
import numpy as np
import datetime
import imageio
from pathlib import Path
from typing import List
from dataclasses import dataclass
from utils.utils import prctile_norm
from models import twostage_Unet


@dataclass
class TestConfig:
    model: any
    input_y: int
    input_x: int
    insert_xy: int
    upsample_flag: bool
    weights_path: Path


def load_image(image_path: str, insert_xy: int) -> np.ndarray:
    img = np.array(imageio.mimread(image_path), dtype=np.float32)
    img = np.squeeze(np.sum(img, axis=0))
    img = prctile_norm(img)

    # symmetric padding
    pad_x = np.zeros((insert_xy, img.shape[1]), dtype=np.float32)
    pad_y = np.zeros((img.shape[0] + 2 * insert_xy, insert_xy), dtype=np.float32)

    img = np.concatenate((pad_x, img, pad_x), axis=0)
    img = np.concatenate((pad_y, img, pad_y), axis=1)

    return img


def infer_single_image(config: TestConfig, image_path: Path, denoise_dir: Path, decon_dir: Path):
    # Extract frame number from filename
    fname = image_path.stem  # e.g., 'frame_5'
    frame_number = int(fname.split('_')[-1])  # get 5

    padded_img = load_image(str(image_path), config.insert_xy)

    input_shape = (
        1,
        config.input_y + 2 * config.insert_xy,
        config.input_x + 2 * config.insert_xy,
        1,
    )

    padded_img = padded_img.reshape(input_shape)

    pred = config.model.predict(padded_img, verbose=0)
    output1 = np.squeeze(pred[0])
    output2 = np.squeeze(pred[1])

    # Crop
    s = config.insert_xy
    ey, ex = config.input_y, config.input_x

    if config.upsample_flag:
        output2 = output2[2*s:2*(s+ey), 2*s:2*(s+ex)]
    else:
        output2 = output2[s:s+ey, s:s+ex]

    # Normalize to uint16
    output1 = np.uint16(prctile_norm(output1) * 65535)
    output2 = np.uint16(prctile_norm(output2) * 65535)

    # Save
    imageio.imwrite(denoise_dir / f"denoised_{frame_number}.tif", output1)
    imageio.imwrite(decon_dir / f"deconved_{frame_number}.tif", output2)


def main():

    # === Base path ===
    base_path = Path(
        "C:/Users/Nikon_1/Documents/JupyterLab/hansol/ZS-DeconvNet/Python_MATLAB_Codes/your_saved_models/ellipses4000_with_100filaments_kidanPSF_with_Gaussian0p5sigma_twostage_Unet_Hess0.02"
    )

    real_time_test_path = base_path / "real_time_test"
    weight_file = base_path / "weights_1000.h5"

    # === Model config ===
    insert_xy, input_y, input_x = 16, 912, 1984
    upsample_flag = True

    print("Building model...")
    model = twostage_Unet.Unet(
        (input_y + 2 * insert_xy, input_x + 2 * insert_xy, 1),
        upsample_flag=upsample_flag,
        insert_x=insert_xy,
        insert_y=insert_xy,
    )

    print("Loading weights...")
    model.load_weights(weight_file)

    config = TestConfig(
        model=model,
        input_y=input_y,
        input_x=input_x,
        insert_xy=insert_xy,
        upsample_flag=upsample_flag,
        weights_path=weight_file,
    )

    # === Find all row folders ===
    row_folders = sorted(real_time_test_path.glob("row_*"))

    if not row_folders:
        print("No row folders found.")
        return

    print(f"Found {len(row_folders)} row folders.")

    total_start = datetime.datetime.now()

    for row_path in row_folders:

        raw_path = row_path / "raw"
        if not raw_path.exists():
            print(f"Skipping {row_path.name} (no raw folder)")
            continue

        inferred_path = row_path / "inferred"
        denoise_path = inferred_path / "denoise"
        decon_path = inferred_path / "decon"
        denoise_path.mkdir(parents=True, exist_ok=True)
        decon_path.mkdir(parents=True, exist_ok=True)

        #image_files = sorted(raw_path.glob("frame_*.tiff"))
        image_files = sorted(raw_path.glob("frame_*.tif"))
        if not image_files:
            print(f"No TIFF files in {raw_path}")
            continue

        print(f"\nProcessing {row_path.name} ({len(image_files)} frames)")

        for image_file in image_files:
            infer_single_image(config, image_file, denoise_path, decon_path)


        print("\nAll inference completed.")
        print(f"Total time: {datetime.datetime.now() - total_start}")


if __name__ == "__main__":
    main()


TensorFlow version: 2.5.0
Built with CUDA: True
GPU devices available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TF built with CUDA: 64_112
Building model...
Loading weights...
Found 1 row folders.

Processing row_001 (51 frames)

All inference completed.
Total time: 0:01:08.790276
